# Fine-tune GPT-2 Vietnamese for Math Word Problems — **V2**

**Improvements over baseline:**
- Data cleaning + answer normalization (one universal anchor: `Đáp án là: <num>`)
- Anchor-safe truncation (always keep `Đáp án là: …` in target)
- `MAX_LENGTH=768` (covers ~99.4% of records without truncation)
- Decoding: beam search + `no_repeat_ngram_size` + `repetition_penalty` + custom `StoppingCriteria`
- Robust answer extraction (case-insensitive, multi-anchor fallback)
- Optional **self-consistency** (sampling + majority vote)
- Optional **type-aware few-shot** prompting

**Pipeline:** load → clean → SFT → generate → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, runtime ≤ 3 h. Backbone fixed to `NlpHUST/gpt2-vietnamese`.


In [ ]:
import os, sys, json, math, time, re, random, hashlib, inspect, unicodedata
from collections import Counter
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    str(Path("dataset")),                 # local fallback
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    str(Path("GPT2_vietnamese")),         # local fallback
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"      # Phase 2

# Run mode: "phase1" -> infer on valid;  "phase2" -> infer on test
RUN_MODE = "phase1"

# Prompt & special tokens
PROMPT_TEMPLATE = "Bài toán: {q}\nLời giải: "
ANSWER_SUFFIX_TEMPLATE = "\nĐáp án là: {a}"     # always appended after normalization
SAFE_EOS_ID = 50256

# Working dirs
WORKING_DIR        = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR         = WORKING_DIR / "gpt2_math_ckpt_v2"
VALID_OUTPUT_PATH  = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH  = WORKING_DIR / "valid_report.json"
TEST_OUTPUT_PATH   = WORKING_DIR / "test_predictions.json"

# Data
MAX_TRAIN_SAMPLES = None             # set to e.g. 5000 for a quick dry-run
MAX_VALID_SAMPLES = None
FILTER_DUPLICATE_QUERIES = False     # legacy flag: drop records that share the same query_vi
                                     #   WARNING: in train.json, ~41% of records share a query with another
                                     #   record but have a DIFFERENT response (paraphrase / alt. CoT).
                                     #   Setting this True throws away high-quality augmentation. Keep False.
DROP_EXACT_DUPLICATES = True         # safer: drop only when BOTH query_vi AND response_vi match (a handful)
DROP_NON_EXTRACTABLE = True          # drop train records whose gold has no numeric answer
NORMALIZE_TARGET_FORMAT = True       # force "...\nĐáp án là: <num>" at the end of every response
KEEP_ORIGINAL_REASONING = True       # if True, keep original solution text, only fix the tail

# Lengths
# GPT-2 has absolute positional embeddings of size N_POSITIONS=1024. The HARD
# constraint at inference is: prompt_len + new_tokens <= N_POSITIONS, else a
# CUDA assert fires from the position-embedding lookup. We size MAX_LENGTH
# (for training) and MAX_NEW_TOKENS (for inference) so they NEVER conflict.
N_POSITIONS       = 1024             # gpt2-vietnamese hard limit
MAX_LENGTH        = 768              # 99.4% of train records fit; baseline used 512
MAX_NEW_TOKENS    = 320              # 320 + 700 prompt slack < 1024; covers p95 response

# Training
EPOCHS                  = 1          # bumped from baseline's 1; raise to 2 if time budget allows
PER_DEVICE_BATCH_SIZE   = 4
GRAD_ACCUM              = 8
LR                      = 5e-5
WARMUP_RATIO            = 0.05
WEIGHT_DECAY            = 0.01
SEED                    = 42

# Decoding
DECODE_MODE             = "beam"     # "greedy" | "beam" | "self_consistency"
NUM_BEAMS               = 4
NO_REPEAT_NGRAM         = 4
REPETITION_PENALTY      = 1.2
LENGTH_PENALTY          = 0.9
# self-consistency only
SC_N                    = 5
SC_TEMPERATURE          = 0.7
SC_TOP_P                = 0.9

# Inference safety knobs
INFER_FP16              = True       # half precision at inference

USE_TYPE_AWARE_FEWSHOT  = False      # enable simple few-shot prepend per type at inference

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

print("TRAIN_FILE :", TRAIN_FILE)
print("VALID_FILE :", VALID_FILE)
print("MODEL_NAME :", MODEL_NAME)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("RUN_MODE   :", RUN_MODE)


In [ ]:
# ============================================================
# 2. Tokenizer smoke-check
# ============================================================
def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # The provided tokenizer maps id 50256 to a normal token string ("hue")
    # even though the task requires using 50256 as EOS/PAD. Strip it manually.
    return tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

probe = "Bài toán: 2+3=?\nLời giải: "
print("Vocab size:", tokenizer.vocab_size)
print("Token ids :", tokenizer(probe, add_special_tokens=False)["input_ids"][:20], "...")
print("SAFE_EOS raw decode:", repr(tokenizer.decode([SAFE_EOS_ID])))
print("SAFE_EOS stripped  :", repr(decode_model_text(tokenizer, [16, SAFE_EOS_ID])))


In [ ]:
# ============================================================
# 3. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records))
print("first query:", train_records[0]["query_vi"][:160])
print("first type :", train_records[0].get("type"))


In [ ]:
# ============================================================
# 4. Answer extraction + data cleaning + target normalization
# ============================================================

# ---- regex pool (case-insensitive, full-width tolerant) ----
RE_ANCHORS_VI = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
]
RE_ANCHORS_EN = [
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
RE_BOXED = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

# Numeric literal patterns
RE_NUM_VI_DEC = re.compile(r"-?\d+,\d+")          # 1,5
RE_NUM_EN_DEC = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")
RE_NUM_LOOSE  = re.compile(r"-?\d+(?:[.,]\d+)?")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}

def _clean_tail(s: str) -> str:
    s = s.strip()
    # take first non-empty line after anchor
    s = s.split("\n", 1)[0].strip()
    # strip trailing punctuation / currency
    s = re.sub(r"[.,;:。、,]+$", "", s)
    # drop "đô la", "USD", "%" trailing units
    s = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", s, flags=re.IGNORECASE)
    s = s.strip()
    return s

def extract_anchor_answer(text: str | None) -> str | None:
    if not text:
        return None
    # last hit per anchor wins (final answer is usually at the end)
    best_pos, best_tail = -1, None
    for pat in RE_ANCHORS_VI + RE_ANCHORS_EN:
        for m in pat.finditer(text):
            if m.end() > best_pos:
                best_pos = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    # boxed fallback (take LAST boxed)
    boxes = RE_BOXED.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(s: str | None) -> float | None:
    if s is None:
        return None
    t = s.strip()
    if not t:
        return None
    # pure VI decimal
    if RE_NUM_VI_DEC.fullmatch(t):
        try:
            v = float(t.replace(",", "."))
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    if RE_NUM_EN_DEC.fullmatch(t):
        try:
            v = float(t)
            return v if math.isfinite(v) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")
    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")
    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)
    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    try:
        v = eval(t.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = float(v)
        return v if math.isfinite(v) else None
    return None

def extract_gold(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("response_vi"))
    return s, parse_number(s)

def extract_pred(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("model_output"))
    return s, parse_number(s)

# ---- normalize target: keep reasoning, force final anchor ----
RE_TRAILING_ANCHORS = re.compile(
    r"(\s*(####\s*[-\d., ]*|"
    r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?[^\n]*|"
    r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?[^\n]*|"
    r"the\s*answer\s*is\s*[:：]?[^\n]*))+\s*$",
    re.IGNORECASE,
)

def normalize_response(resp: str, gold_num: float | None, gold_str: str | None) -> str:
    """Strip trailing answer-anchors, then append a single canonical anchor."""
    body = RE_TRAILING_ANCHORS.sub("", resp.rstrip()).rstrip()
    # decide the canonical answer string
    if gold_num is not None:
        if gold_num == int(gold_num):
            canonical = str(int(gold_num))
        else:
            # use the original gold string if it looks well-formed, else float repr
            canonical = gold_str if gold_str and re.fullmatch(r"-?\d+(?:[.,]\d+)?", gold_str) else f"{gold_num:g}"
    else:
        canonical = gold_str if gold_str else ""
    return f"{body}\nĐáp án là: {canonical}"

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen = set()
    out = []
    dropped_no_ans = 0
    dropped_dup = 0
    normalized = 0

    for rec in records:
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_no_ans += 1
            continue
        if split == "train":
            if FILTER_DUPLICATE_QUERIES:
                key = q
            elif DROP_EXACT_DUPLICATES:
                key = (q, r)
            else:
                key = None
            if key is not None:
                if key in seen:
                    dropped_dup += 1
                    continue
                seen.add(key)

        gold_str, gold_num = extract_gold(rec)
        if DROP_NON_EXTRACTABLE and split == "train" and gold_num is None:
            dropped_no_ans += 1
            continue

        new_r = r
        if NORMALIZE_TARGET_FORMAT and split == "train" and KEEP_ORIGINAL_REASONING:
            new_r = normalize_response(r, gold_num, gold_str)
            if new_r != r:
                normalized += 1

        out.append({
            **rec,
            "query_vi": q,
            "response_vi": new_r,
            "_gold_num": gold_num,
            "_gold_str": gold_str,
        })

    print(f"[{split}] kept={len(out)} dropped_dup={dropped_dup} "
          f"dropped_no_ans={dropped_no_ans} normalized={normalized}")
    return out

train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid")  # no filtering, just gold extraction

print("\nExample BEFORE / AFTER:")
ex = train_records[0]
print("BEFORE:", ex["response_vi"][-200:])
print("AFTER :", train_clean[0]["response_vi"][-200:])


In [ ]:
# ============================================================
# 5. SFT dataset (anchor-safe truncation) + collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response); mask loss on prompt + padding.

    Truncation policy: if the full sequence exceeds max_length, we drop
    from the *middle* of the response so that the final
    `\nĐáp án là: <num>\u200b<eos>` always survives.
    """

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"])
        response = rec["response_vi"]

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        budget = self.max_length - len(p_ids)
        if budget <= 4:
            # prompt itself too long → truncate from left of prompt
            p_ids = p_ids[-(self.max_length - 8):]
            budget = self.max_length - len(p_ids)

        if len(r_ids) > budget:
            # keep the last `tail_keep` tokens (carries the anchor + answer + EOS)
            tail_keep = min(96, budget // 2)
            head_keep = budget - tail_keep
            r_ids = r_ids[:head_keep] + r_ids[-tail_keep:]

        ids    = p_ids + r_ids
        labels = [-100] * len(p_ids) + r_ids
        # Defensive clamp: never feed out-of-range ids.
        ids    = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

# quick sanity check
_ds_probe = SFTDataset(train_clean[:1], tokenizer, MAX_LENGTH)
_sample = _ds_probe[0]
print("sample len:", len(_sample["input_ids"]),
      "| n_loss_tokens:", sum(1 for x in _sample["labels"] if x != -100))
print("last 15 tokens:", decode_model_text(tokenizer, _sample["input_ids"][-15:]))


In [ ]:
# ============================================================
# 6. Evaluation utilities (uses the upgraded extractors above)
# ============================================================
def rel_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(re_val, is_extractable):
    if not is_extractable or re_val is None:
        return 0
    if re_val <= 0.01: return 10
    if re_val <= 0.10: return 5
    if re_val <= 0.50: return 1
    return 0

def evaluate(pred_items, gold_items):
    assert len(pred_items) == len(gold_items), (len(pred_items), len(gold_items))
    rows = []
    total = 0
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in zip(pred_items, gold_items):
        gold_str, gold_num = extract_gold(gold_rec)
        pred_str, pred_num = extract_pred(pred_rec)

        is_extractable = pred_str is not None
        extractable += int(is_extractable)
        re_val = rel_error(pred_num, gold_num)
        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s
        buckets[s] = buckets.get(s, 0) + 1

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_str, "gold_num": gold_num,
            "pred_answer": pred_str, "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": n * 10,
            "score_10": total / n if n else 0.0,
            "score_pct": (total / (n * 10)) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }

def save_evaluation_report(pred_path, gold_records, report_path):
    pred_path = Path(pred_path); report_path = Path(report_path)
    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)
    result = evaluate(pred_items, gold_records)
    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Wrote {report_path}")
    return result


In [ ]:
# ============================================================
# 7. StoppingCriteria + decoding
# ============================================================
class StopOnAnswerLine(StoppingCriteria):
    """Stop when generation has emitted a final `Đáp án là: <number>` AND the
    next token starts a new line or EOS. Cheap heuristic: stop when we have
    decoded a substring matching the answer-line regex with a trailing newline
    or when EOS appears.
    """
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID,
                 patience_tokens: int = 24):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None     # token index when first matched

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        # decode only the generated tail (cheaper)
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 6:
            return False
        text = self.tok.decode(gen_tail, skip_special_tokens=True)
        m = self.re_answer.search(text)
        if m:
            if self._matched_at is None:
                self._matched_at = gen_tail.numel()
            # let it spit out a few more digits, then stop on newline or patience
            if "\n" in text[m.end():]:
                return True
            if gen_tail.numel() - self._matched_at >= self.patience:
                return True
        return False


def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())


# ---- optional type-aware few-shot exemplars ----
FEWSHOT_BY_TYPE = {
    "GSM_AnsAug": (
        "Bài toán: Lan có 5 quả cam, mẹ cho thêm 3 quả. Hỏi Lan có bao nhiêu quả cam?\n"
        "Lời giải: Lan ban đầu có 5 quả. Mẹ cho thêm 3 quả. Tổng cộng 5 + 3 = 8 quả.\n"
        "Đáp án là: 8\n\n"
    ),
    "GSM_Rephrased": (
        "Bài toán: Nếu Nam có 10 viên kẹo và ăn 4 viên thì còn bao nhiêu?\n"
        "Lời giải: Nam ăn 4 trên 10 viên kẹo nên còn 10 - 4 = 6 viên.\n"
        "Đáp án là: 6\n\n"
    ),
    "MATH_AnsAug": (
        "Bài toán: Tính giá trị của $2^5$.\n"
        "Lời giải: $2^5 = 2 \\cdot 2 \\cdot 2 \\cdot 2 \\cdot 2 = 32$.\n"
        "Đáp án là: 32\n\n"
    ),
    "MATH_Rephrased": (
        "Bài toán: Tìm $x$ thỏa mãn $3x = 12$.\n"
        "Lời giải: Chia hai vế cho 3, ta có $x = 12/3 = 4$.\n"
        "Đáp án là: 4\n\n"
    ),
    "GSM_FOBAR": (
        "Bài toán: An có x quả bóng. An cho 2 quả. Còn lại 5 quả. "
        "Nếu biết câu trả lời là 5 thì x bằng bao nhiêu?\n"
        "Lời giải: x - 2 = 5 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "GSM_SV": (
        "Bài toán: Bình có x cái bút. Bình cho bạn 3 cái, còn 4 cái. Tìm x.\n"
        "Lời giải: x - 3 = 4 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "MATH_FOBAR": (
        "Bài toán: $f(x) = 2x + X$. Nếu $f(3) = 7$ thì $X$ bằng bao nhiêu?\n"
        "Lời giải: $2 \\cdot 3 + X = 7$ nên $X = 1$.\n"
        "Đáp án là: 1\n\n"
    ),
    "MATH_SV": (
        "Bài toán: $X + 2 = 5$. Tìm $X$.\n"
        "Lời giải: $X = 5 - 2 = 3$.\n"
        "Đáp án là: 3\n\n"
    ),
}

def build_prompt_with_fewshot(rec: dict) -> str:
    if not USE_TYPE_AWARE_FEWSHOT:
        return build_prompt(rec)
    fs = FEWSHOT_BY_TYPE.get(rec.get("type"), "")
    return fs + build_prompt(rec)


def _vote_answer(cands: list[str | None]) -> str | None:
    nums = []
    for c in cands:
        n = parse_number(extract_anchor_answer(c))
        if n is not None:
            nums.append((round(n, 6), c))
    if not nums:
        return cands[0] if cands else None
    counter = Counter(n for n, _ in nums)
    top_num, _ = counter.most_common(1)[0]
    for n, c in nums:
        if n == top_num:
            return c
    return cands[0]


@torch.inference_mode()
def generate_outputs(model_path_or_name, records, output_path,
                     max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[infer] Loading {model_path_or_name} on {device} (mode={mode}) ...", flush=True)

    tok = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tok.pad_token_id = SAFE_EOS_ID
    tok.eos_token_id = SAFE_EOS_ID
    if tok.padding_side != "left":
        tok.padding_side = "left"   # left-pad for batched generation

    dtype = torch.float16 if (device == "cuda" and INFER_FP16) else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name, torch_dtype=dtype, local_files_only=True,
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()
    vocab_n = model.transformer.wte.num_embeddings
    # GPT-2 absolute positional embedding hard limit. Crossing this triggers a
    # CUDA assert (`indexSelectSmallIndex: srcIndex < srcSelectDimSize`) inside
    # the position-embedding lookup. We MUST keep prompt_len + new_tokens <= n_pos.
    n_pos = int(getattr(model.config, "n_positions",
                        getattr(model.config, "max_position_embeddings", 1024)))

    outputs, t0 = [], time.time()
    for idx, rec in enumerate(tqdm(records, desc=f"gen[{mode}]")):
        prompt = build_prompt_with_fewshot(rec)

        # ---- Budget-aware tokenization ----
        # Leave room for `max_new_tokens` new tokens inside the 1024 position cap.
        # If the prompt is too long, truncate from the LEFT so the trailing
        # "Lời giải:" anchor is preserved (the answer cue MUST stay at the end).
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        # safety clamp on token ids (tokenizer.vocab may > model embedding)
        full_ids = [min(t, vocab_n - 1) for t in full_ids]
        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]

        # Final safety clamp: never let new_tokens push past n_pos.
        eff_new = max(8, min(max_new_tokens, n_pos - prompt_len))

        common_kwargs = dict(
            input_ids=ids, attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID, eos_token_id=SAFE_EOS_ID,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            repetition_penalty=REPETITION_PENALTY,
            stopping_criteria=StoppingCriteriaList([
                StopOnAnswerLine(tok, prompt_len=prompt_len)
            ]),
        )

        if mode == "greedy":
            gen = model.generate(do_sample=False, num_beams=1, **common_kwargs)
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "beam":
            gen = model.generate(
                do_sample=False, num_beams=NUM_BEAMS,
                length_penalty=LENGTH_PENALTY, early_stopping=True,
                **common_kwargs,
            )
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "self_consistency":
            cands = []
            for s in range(SC_N):
                gen = model.generate(
                    do_sample=True, temperature=SC_TEMPERATURE, top_p=SC_TOP_P,
                    num_beams=1, **common_kwargs,
                )
                cands.append(decode_model_text(tok, gen[0, prompt_len:]))
            text = _vote_answer(cands) or cands[0]
        else:
            raise ValueError(mode)

        outputs.append({
            "id": idx,
            "query_vi": rec["query_vi"],
            "type": rec.get("type"),
            "model_output": text,
        })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)
    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"[infer] Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    # free
    del model
    torch.cuda.empty_cache()
    return outputs


In [ ]:
# ============================================================
# 8. Train
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
model.gradient_checkpointing_enable()
model.config.use_cache = False

train_ds = SFTDataset(train_clean, tokenizer, MAX_LENGTH)
# tiny eval sub-sample to keep eval fast
EVAL_SUB = min(200, len(valid_clean))
valid_ds = SFTDataset(valid_clean[:EVAL_SUB], tokenizer, MAX_LENGTH)
collator = PadCollator(pad_id=SAFE_EOS_ID)

eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
print(f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} "
      f"| gpus={torch.cuda.device_count()} | eff_batch={eff_batch} "
      f"| train_size={len(train_ds)} | steps/epoch={steps_per_epoch}")

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "epoch"
else:
    ta_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

t0 = time.time()
trainer.train()
train_dt = time.time() - t0
print(f"\n[train] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model_hash = sha256_dir(OUTPUT_DIR)
(OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
print("Saved checkpoint to:", OUTPUT_DIR, "| SHA256:", model_hash)

del trainer, model
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 9. Inference + evaluation
# ============================================================
if RUN_MODE == "phase1":
    valid_outputs = generate_outputs(
        OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
    )
    print("\nExample output:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

    valid_result = save_evaluation_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
    s = valid_result["summary"]
    print("\nFinal validation score:")
    print(f'{s["raw_score"]} / {s["max_raw_score"]}  ({s["score_pct"]*100:.2f}%)')
    print(f'Score /10: {s["score_10"]:.4f}')
    print("Buckets:", s["buckets"])
else:
    print("Skipping phase1 inference (RUN_MODE != phase1).")


In [ ]:
# ============================================================
# 10. Error analysis preview
# ============================================================
if RUN_MODE == "phase1":
    rows = valid_result["rows"]
    by_type = {}
    for r in rows:
        t = r.get("type") or "UNK"
        by_type.setdefault(t, []).append(r)

    print("Per-type score breakdown:")
    print(f"{'type':<22s} {'n':>4s} {'mean':>6s} {'b10':>4s} {'b5':>4s} {'b1':>4s} {'b0':>4s} {'extr%':>6s}")
    for t, rs in sorted(by_type.items()):
        n = len(rs)
        mean_s = sum(r["score"] for r in rs) / n
        b10 = sum(r["score"] == 10 for r in rs)
        b5  = sum(r["score"] == 5 for r in rs)
        b1  = sum(r["score"] == 1 for r in rs)
        b0  = sum(r["score"] == 0 for r in rs)
        extr = 100 * sum(bool(r["extractable"]) for r in rs) / n
        print(f"{t:<22s} {n:>4d} {mean_s:>6.2f} {b10:>4d} {b5:>4d} {b1:>4d} {b0:>4d} {extr:>5.1f}%")

    bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
    print(f"\n{len(bad)} zero-score samples. Showing first 2:")
    for i, r in bad[:2]:
        pred = valid_outputs[i]; gold = valid_records[i]
        print("=" * 90)
        print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
        print("QUERY:", gold["query_vi"][:400])
        print("GOLD :", gold["response_vi"][-300:])
        print("PRED :", pred["model_output"][:800])


In [ ]:
# ============================================================
# 11. Phase 2 — generate test_predictions.json
# ============================================================
# Set RUN_MODE = "phase2" at the top to activate this block.
if RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"Cannot find test.json at {TEST_FILE}")
    test_records = load_records(TEST_FILE)
    print("test:", len(test_records))

    test_outputs = generate_outputs(
        OUTPUT_DIR, test_records, TEST_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
    )

    # Required schema check
    required_keys = {"id", "query_vi", "type", "model_output"}
    missing = [k for k in required_keys if k not in test_outputs[0]]
    if missing:
        raise ValueError(f"Output missing keys: {missing}")

    print("First test prediction:")
    print(json.dumps(test_outputs[0], ensure_ascii=False, indent=2)[:1500])


In [ ]:
# ============================================================
# 12. List saved artifacts
# ============================================================
candidates = [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    OUTPUT_DIR / "model_hash.txt",
    TEST_OUTPUT_PATH,
    Path(str(TEST_OUTPUT_PATH) + ".sha256.txt"),
]
for p in candidates:
    p = Path(p)
    print(p, "| exists =", p.exists(), "| size =", p.stat().st_size if p.exists() else None)

print("\nWorking dir contents:")
for p in sorted(Path(WORKING_DIR).glob("*")):
    print("-", p)
